# NDVI Processing with Multi-band Imagery

This notebook demonstrates vegetation analysis using NDVI (Normalized Difference Vegetation Index) with RGBI imagery, digital surface models (DSM), and digital terrain models (DEM).

**Key concepts covered:**
- 🌿 **NDVI calculation** from Red-Green-Blue-Infrared (RGBI) imagery
- 🏔️ **Height analysis** using DSM and DEM data
- 🗻 **Slope analysis** for terrain characteristics
- 🎯 **Vegetation classification** combining spectral and height information
- 📊 **Multi-sensor data fusion** techniques

## Prerequisites

Install required dependencies:
```bash
pip install rasterio numpy opencv-python scikit-image shapely scipy gdal
```

## 1. Import Dependencies and Setup

In [ ]:
import os
import setup_gdal_env  # Configure GDAL environment
import numpy as np
import rasterio
from rasterio.windows import from_bounds
import matplotlib.pyplot as plt
from scipy.ndimage import convolve, binary_closing
from skimage.transform import resize
from shapely.geometry import Point, Polygon
import cv2

# Optional GDAL for advanced slope processing
try:
    from osgeo import gdal
    GDAL_AVAILABLE = True
    print("✅ GDAL available for advanced slope processing")
except ImportError:
    GDAL_AVAILABLE = False
    print("⚠️  GDAL not available. Basic slope processing only.")

print("✅ Dependencies imported successfully!")
print("📊 Ready for NDVI processing")

## 2. NDVI Processor Class Definition

In [ ]:
class NDVIProcessor:
    """A comprehensive NDVI and vegetation analysis processor."""
    
    def __init__(self):
        self.use_height = True
        self.in_debug_mode = False
        print("🌿 NDVIProcessor initialized")
        print(f"   Height analysis: {'Enabled' if self.use_height else 'Disabled'}")
        print(f"   Debug mode: {'On' if self.in_debug_mode else 'Off'}")

    def set_height_usage(self, use_height: bool):
        """Sets whether to use height information in vegetation classification."""
        self.use_height = use_height
        print(f"🏔️  Height analysis: {'Enabled' if use_height else 'Disabled'}")

    def set_debug_mode(self, debug_mode: bool):
        """Sets the debug mode for the processor."""
        self.in_debug_mode = debug_mode
        print(f"🐛 Debug mode: {'On' if debug_mode else 'Off'}")

    def calculate_ndvi(self, rgbi: np.ndarray) -> np.ndarray:
        """Calculate NDVI from RGBI imagery.
        
        NDVI = (NIR - Red) / (NIR + Red)
        Where NIR (Near Infrared) is typically band 4 and Red is band 1
        """
        red = rgbi[0].astype(np.float32)   # Red band
        nir = rgbi[3].astype(np.float32)   # Near-infrared band
        
        # Avoid division by zero
        denominator = nir + red
        denominator = np.where(denominator == 0, 1, denominator)
        
        ndvi = (nir - red) / denominator
        
        # Clip to valid NDVI range [-1, 1]
        ndvi = np.clip(ndvi, -1, 1)
        
        if self.in_debug_mode:
            print(f"📊 NDVI Statistics:")
            print(f"   Min: {ndvi.min():.3f}")
            print(f"   Max: {ndvi.max():.3f}")
            print(f"   Mean: {ndvi.mean():.3f}")
            print(f"   Std: {ndvi.std():.3f}")
        
        return ndvi
    
    def calculate_ndwi(self, rgbi: np.ndarray) -> np.ndarray:
        """Calculate NDWI (Normalized Difference Water Index) from RGBI imagery.
        
        NDWI = (Green - NIR) / (Green + NIR)
        """
        green = rgbi[1].astype(np.float32)  # Green band
        nir = rgbi[3].astype(np.float32)    # Near-infrared band
        
        # Avoid division by zero
        denominator = green + nir
        denominator = np.where(denominator == 0, 1, denominator)
        
        ndwi = (green - nir) / denominator
        
        # Clip to valid range [-1, 1]
        ndwi = np.clip(ndwi, -1, 1)
        
        return ndwi

print("✅ NDVIProcessor class defined!")

## 3. File Reading Methods

In [ ]:
class NDVIProcessor(NDVIProcessor):  # Extend the existing class
    
    def read_rgbi(self, rgbi_path: str) -> tuple[np.ndarray, rasterio.Affine, tuple]:
        """Reads an RGBI raster file and returns the image array, affine transform, and bounds."""
        print(f"📖 Reading RGBI from: {os.path.basename(rgbi_path)}")
        
        with rasterio.open(rgbi_path, "r") as rgbi_f:
            if self.in_debug_mode:
                print(f"   CRS: {rgbi_f.crs}")
                print(f"   Shape: {rgbi_f.shape}")
                print(f"   Bands: {rgbi_f.count}")
                print(f"   Data type: {rgbi_f.dtypes[0]}")
            
            rgbi_transform = rgbi_f.transform
            rgbi_bounds = rgbi_f.bounds
            rgbi = rgbi_f.read().astype(np.float32)
            
        print(f"   ✅ RGBI shape: {rgbi.shape}")
        return rgbi, rgbi_transform, rgbi_bounds

    def read_dsm(self, dsm_path: str, bounds: tuple = None) -> np.ndarray:
        """Reads a DSM raster file, crops it to the given bounds, and returns the array."""
        print(f"📖 Reading DSM from: {os.path.basename(dsm_path)}")
        
        if bounds is None:
            with rasterio.open(dsm_path) as dsm_f:
                dsm = dsm_f.read().astype(np.float32)
        else:
            with rasterio.open(dsm_path, "r") as dsm_f:
                window = rasterio.windows.from_bounds(*bounds, transform=dsm_f.transform)
                dsm = dsm_f.read(window=window).astype(np.float32)
        
        print(f"   ✅ DSM shape: {dsm.shape}")
        if self.in_debug_mode:
            print(f"   Height range: {dsm.min():.1f} to {dsm.max():.1f} meters")
        
        return dsm

    def read_dem(self, dem_path: str, bounds: tuple = None) -> tuple[np.ndarray, rasterio.Affine]:
        """Reads a DEM raster file, crops it to the given bounds, and returns the array."""
        print(f"📖 Reading DEM from: {os.path.basename(dem_path)}")
        
        transform = None
        if bounds is None:
            with rasterio.open(dem_path, "r") as dem_f:
                dem = dem_f.read().astype(np.float32)
                transform = dem_f.transform
        else:
            with rasterio.open(dem_path, "r") as dem_f:
                window = rasterio.windows.from_bounds(*bounds, transform=dem_f.transform)
                dem = dem_f.read(window=window).astype(np.float32)
                transform = dem_f.transform
        
        print(f"   ✅ DEM shape: {dem.shape}")
        if self.in_debug_mode:
            print(f"   Elevation range: {dem.min():.1f} to {dem.max():.1f} meters")
        
        return dem, transform
    
    def read_raster_datasets(self, rgbi_path: str, dsm_path: str = None, dem_path: str = None, 
                           resize_rgbi_base: bool = True) -> tuple[np.ndarray, np.ndarray, np.ndarray, rasterio.Affine]:
        """Reads RGBI, DSM, and DEM raster files and aligns them."""
        print("🔄 Reading and aligning raster datasets...")
        
        # Always read RGBI
        rgbi, transform, bounds = self.read_rgbi(rgbi_path)
        
        dsm = None
        dem = None
        
        if self.use_height and dsm_path and dem_path:
            # Read DSM and DEM
            dsm = self.read_dsm(dsm_path, bounds)
            dem, _ = self.read_dem(dem_path, bounds)
            
            # Resize to match RGBI if needed
            if resize_rgbi_base:
                target_shape = rgbi.shape[1:]  # Height, width from RGBI
                if dsm.shape[1:] != target_shape:
                    print(f"   📏 Resizing DSM from {dsm.shape[1:]} to {target_shape}")
                    dsm_resized = resize(dsm[0], target_shape, preserve_range=True)
                    dsm = dsm_resized[np.newaxis, :, :]  # Add band dimension back
                
                if dem.shape[1:] != target_shape:
                    print(f"   📏 Resizing DEM from {dem.shape[1:]} to {target_shape}")
                    dem_resized = resize(dem[0], target_shape, preserve_range=True)
                    dem = dem_resized[np.newaxis, :, :]  # Add band dimension back
        
        print("✅ Dataset alignment complete")
        return rgbi, dsm, dem, transform

print("✅ File reading methods added!")

## 4. Height and Slope Analysis Methods

In [ ]:
class NDVIProcessor(NDVIProcessor):  # Continue extending
    
    def calculate_ndsm(self, dsm: np.ndarray, dem: np.ndarray) -> np.ndarray:
        """Calculates the normalized Digital Surface Model (nDSM) by subtracting DEM from DSM."""
        print("📐 Calculating normalized DSM (nDSM)...")
        
        ndsm = (dsm[0] - dem[0])
        
        # Set negative values to 0 (where DEM is higher than DSM)
        negative_count = (ndsm < 0).sum()
        ndsm[ndsm < 0] = 0
        
        if self.in_debug_mode:
            print(f"   Negative values corrected: {negative_count} pixels")
            print(f"   nDSM height range: 0 to {ndsm.max():.1f} meters")
            print(f"   Mean object height: {ndsm.mean():.1f} meters")
        
        print(f"   ✅ nDSM calculated (max height: {ndsm.max():.1f}m)")
        return ndsm
    
    def calculate_slope(self, dem: np.ndarray, transform: rasterio.Affine) -> np.ndarray:
        """Calculates slope using Horn's method."""
        print("⛰️  Calculating slope using Horn's method...")
        
        # Extract first band if multi-band
        if len(dem.shape) > 2:
            dem_2d = dem[0]
        else:
            dem_2d = dem
        
        cellsize_x = abs(transform.a)
        cellsize_y = abs(transform.e)
        
        # Horn's method kernels
        kernel_x = np.array([[-1, 0, 1],
                            [-2, 0, 2],
                            [-1, 0, 1]]) / (8 * cellsize_x)

        kernel_y = np.array([[ 1,  2,  1],
                            [ 0,  0,  0],
                            [-1, -2, -1]]) / (8 * cellsize_y)

        # Compute gradients
        dzdx = convolve(dem_2d, kernel_x, mode='nearest')
        dzdy = convolve(dem_2d, kernel_y, mode='nearest')

        # Compute slope in degrees
        slope_rad = np.arctan(np.sqrt(dzdx**2 + dzdy**2))
        slope_deg = np.degrees(slope_rad)
        
        if self.in_debug_mode:
            print(f"   Slope range: {slope_deg.min():.1f}° to {slope_deg.max():.1f}°")
            print(f"   Mean slope: {slope_deg.mean():.1f}°")
        
        print(f"   ✅ Slope calculated (max: {slope_deg.max():.1f}°)")
        return slope_deg
    
    def classify_slope(self, slope: np.ndarray, threshold: float = 5.0) -> np.ndarray:
        """Classify terrain as flat or sloped based on threshold."""
        print(f"🏔️  Classifying slope (threshold: {threshold}°)...")
        
        classification = np.where(slope < threshold, 0, 1)  # 0=flat, 1=sloped
        
        flat_percent = (classification == 0).sum() / classification.size * 100
        sloped_percent = 100 - flat_percent
        
        print(f"   📊 Terrain classification:")
        print(f"      Flat (<{threshold}°): {flat_percent:.1f}%")
        print(f"      Sloped (≥{threshold}°): {sloped_percent:.1f}%")
        
        return classification

print("✅ Height and slope analysis methods added!")

## 5. Vegetation Classification Methods

In [ ]:
class NDVIProcessor(NDVIProcessor):  # Continue extending
    
    def classify_vegetation(self, ndvi: np.ndarray, ndsm: np.ndarray = None, 
                          ndvi_threshold: float = 0.3, height_threshold: float = 3.5,
                          max_height: float = None) -> np.ndarray:
        """Classify vegetation using NDVI and optionally height information."""
        print(f"🌿 Classifying vegetation (NDVI threshold: {ndvi_threshold})...")
        
        # Basic NDVI classification
        vegetation_mask = ndvi > ndvi_threshold
        
        if self.use_height and ndsm is not None:
            print(f"   🏔️  Adding height constraints (min: {height_threshold}m)...")
            
            # Height constraints
            height_mask = ndsm > height_threshold
            
            if max_height is not None:
                print(f"      Maximum height: {max_height}m")
                height_mask = height_mask & (ndsm <= max_height)
            
            # Combine NDVI and height
            vegetation_mask = vegetation_mask & height_mask
            
            # Create detailed classification
            classification = np.zeros_like(ndvi, dtype=np.uint8)
            classification[vegetation_mask] = 3  # High vegetation
            classification[(ndvi > ndvi_threshold) & (ndsm <= height_threshold)] = 2  # Low vegetation
            classification[(ndvi <= ndvi_threshold) & (ndsm > height_threshold)] = 1  # Non-vegetation objects
            # 0 remains for ground/low NDVI areas
            
            class_labels = {
                0: "Ground/Bare",
                1: "Non-vegetation objects", 
                2: "Low vegetation",
                3: "High vegetation (trees)"
            }
            
        else:
            # Simple binary classification
            classification = vegetation_mask.astype(np.uint8)
            class_labels = {
                0: "Non-vegetation",
                1: "Vegetation"
            }
        
        # Calculate statistics
        print(f"   📊 Classification results:")
        total_pixels = classification.size
        for class_id, label in class_labels.items():
            count = (classification == class_id).sum()
            percentage = count / total_pixels * 100
            print(f"      {label}: {percentage:.1f}% ({count:,} pixels)")
        
        return classification
    
    def apply_morphological_cleaning(self, classification: np.ndarray, 
                                   iterations: int = 2) -> np.ndarray:
        """Apply morphological operations to clean up the classification."""
        print(f"🧹 Applying morphological cleaning ({iterations} iterations)...")
        
        cleaned = classification.copy()
        
        # Apply binary closing for each class to fill small gaps
        unique_classes = np.unique(classification)
        unique_classes = unique_classes[unique_classes > 0]  # Skip background
        
        for class_id in unique_classes:
            class_mask = classification == class_id
            cleaned_mask = binary_closing(class_mask, iterations=iterations)
            cleaned[cleaned_mask] = class_id
        
        # Calculate change statistics
        changed_pixels = (classification != cleaned).sum()
        change_percentage = changed_pixels / classification.size * 100
        
        print(f"   ✅ Morphological cleaning complete")
        print(f"      Changed pixels: {changed_pixels:,} ({change_percentage:.2f}%)")
        
        return cleaned

print("✅ Vegetation classification methods added!")

## 6. Visualization Functions

In [ ]:
def visualize_results(rgbi: np.ndarray, ndvi: np.ndarray, classification: np.ndarray, 
                     ndsm: np.ndarray = None, slope: np.ndarray = None):
    """Create visualization plots for NDVI processing results."""
    
    # Calculate number of subplots needed
    subplot_count = 3  # RGB, NDVI, Classification
    if ndsm is not None:
        subplot_count += 1
    if slope is not None:
        subplot_count += 1
    
    # Create subplot layout
    if subplot_count <= 3:
        rows, cols = 1, subplot_count
        figsize = (15, 5)
    else:
        rows, cols = 2, 3
        figsize = (18, 12)
    
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = axes.flatten() if subplot_count > 1 else [axes]
    
    plot_idx = 0
    
    # RGB composite
    if rgbi.shape[0] >= 3:
        rgb = np.dstack([rgbi[0], rgbi[1], rgbi[2]])  # R, G, B
        # Normalize to 0-1 range
        rgb_norm = (rgb - rgb.min()) / (rgb.max() - rgb.min())
        rgb_norm = np.clip(rgb_norm, 0, 1)
        
        axes[plot_idx].imshow(rgb_norm)
        axes[plot_idx].set_title('RGB Composite')
        axes[plot_idx].axis('off')
        plot_idx += 1
    
    # NDVI
    ndvi_plot = axes[plot_idx].imshow(ndvi, cmap='RdYlGn', vmin=-1, vmax=1)
    axes[plot_idx].set_title('NDVI')
    axes[plot_idx].axis('off')
    plt.colorbar(ndvi_plot, ax=axes[plot_idx], fraction=0.046, pad=0.04)
    plot_idx += 1
    
    # Vegetation Classification
    class_plot = axes[plot_idx].imshow(classification, cmap='viridis')
    axes[plot_idx].set_title('Vegetation Classification')
    axes[plot_idx].axis('off')
    plt.colorbar(class_plot, ax=axes[plot_idx], fraction=0.046, pad=0.04)
    plot_idx += 1
    
    # nDSM (if available)
    if ndsm is not None and plot_idx < len(axes):
        ndsm_plot = axes[plot_idx].imshow(ndsm, cmap='terrain')
        axes[plot_idx].set_title('Normalized DSM (Height above ground)')
        axes[plot_idx].axis('off')
        plt.colorbar(ndsm_plot, ax=axes[plot_idx], fraction=0.046, pad=0.04, label='Height (m)')
        plot_idx += 1
    
    # Slope (if available)
    if slope is not None and plot_idx < len(axes):
        slope_plot = axes[plot_idx].imshow(slope, cmap='plasma')
        axes[plot_idx].set_title('Slope (degrees)')
        axes[plot_idx].axis('off')
        plt.colorbar(slope_plot, ax=axes[plot_idx], fraction=0.046, pad=0.04, label='Degrees')
        plot_idx += 1
    
    # Hide any unused subplots
    for i in range(plot_idx, len(axes)):
        axes[i].set_visible(False)
    
    plt.tight_layout()
    plt.show()

def create_statistics_summary(ndvi: np.ndarray, classification: np.ndarray, 
                            ndsm: np.ndarray = None):
    """Create a summary of processing statistics."""
    print("\n📈 === Processing Results Summary ===")
    
    # NDVI statistics
    print(f"\n🌿 NDVI Statistics:")
    print(f"   Range: {ndvi.min():.3f} to {ndvi.max():.3f}")
    print(f"   Mean: {ndvi.mean():.3f} ± {ndvi.std():.3f}")
    print(f"   Vegetation pixels (NDVI > 0.3): {(ndvi > 0.3).sum():,} ({(ndvi > 0.3).mean()*100:.1f}%)")
    print(f"   Healthy vegetation (NDVI > 0.5): {(ndvi > 0.5).sum():,} ({(ndvi > 0.5).mean()*100:.1f}%)")
    
    # Classification statistics
    print(f"\n🎯 Classification Statistics:")
    unique_classes = np.unique(classification)
    total_pixels = classification.size
    
    for class_id in unique_classes:
        count = (classification == class_id).sum()
        percentage = count / total_pixels * 100
        print(f"   Class {class_id}: {count:,} pixels ({percentage:.1f}%)")
    
    # Height statistics (if available)
    if ndsm is not None:
        print(f"\n🏔️  Height Statistics:")
        print(f"   Height range: {ndsm.min():.1f} to {ndsm.max():.1f} meters")
        print(f"   Mean height: {ndsm.mean():.1f} ± {ndsm.std():.1f} meters")
        print(f"   Objects > 3m: {(ndsm > 3).sum():,} pixels ({(ndsm > 3).mean()*100:.1f}%)")
        print(f"   Objects > 10m: {(ndsm > 10).sum():,} pixels ({(ndsm > 10).mean()*100:.1f}%)")

print("✅ Visualization functions defined!")

## 7. Practical Example: Processing Sample Data

In [ ]:
# Create sample synthetic data for demonstration
def create_sample_data():
    """Create synthetic RGBI, DSM, and DEM data for demonstration."""
    print("🎭 Creating synthetic sample data for demonstration...")
    
    # Image dimensions
    height, width = 200, 200
    
    # Create coordinate grids
    x = np.linspace(0, 10, width)
    y = np.linspace(0, 10, height)
    X, Y = np.meshgrid(x, y)
    
    # Create synthetic RGBI data
    # Simulate different land cover types
    
    # Forest area (high NIR, moderate red)
    forest_mask = (X > 2) & (X < 6) & (Y > 3) & (Y < 8)
    
    # Grass area (moderate NIR and green)
    grass_mask = (X > 6) & (Y > 2) & (Y < 7)
    
    # Urban/bare area (low NIR)
    urban_mask = ~(forest_mask | grass_mask)
    
    # Initialize bands
    red = np.random.normal(100, 20, (height, width)).clip(0, 255)
    green = np.random.normal(120, 25, (height, width)).clip(0, 255)
    blue = np.random.normal(80, 15, (height, width)).clip(0, 255)
    nir = np.random.normal(90, 20, (height, width)).clip(0, 255)
    
    # Modify values for different land cover types
    # Forest: high NIR, lower red
    nir[forest_mask] = np.random.normal(200, 30, forest_mask.sum()).clip(150, 255)
    red[forest_mask] = np.random.normal(60, 15, forest_mask.sum()).clip(30, 100)
    green[forest_mask] = np.random.normal(100, 20, forest_mask.sum()).clip(60, 140)
    
    # Grass: moderate NIR, higher green
    nir[grass_mask] = np.random.normal(150, 25, grass_mask.sum()).clip(100, 200)
    green[grass_mask] = np.random.normal(150, 20, grass_mask.sum()).clip(100, 200)
    red[grass_mask] = np.random.normal(80, 15, grass_mask.sum()).clip(50, 120)
    
    # Stack into RGBI array
    rgbi = np.stack([red, green, blue, nir]).astype(np.float32)
    
    # Create synthetic DSM (elevation + objects)
    # Base terrain elevation
    base_elevation = 100 + 50 * np.sin(X/2) + 30 * np.cos(Y/3)
    
    # Add buildings and trees
    dsm = base_elevation.copy()
    
    # Trees in forest areas (5-20m high)
    tree_heights = np.random.uniform(5, 20, dsm.shape)
    dsm[forest_mask] += tree_heights[forest_mask]
    
    # Buildings in urban areas (3-15m high)
    building_mask = urban_mask & ((X + Y) % 2 < 0.3)  # Scattered buildings
    building_heights = np.random.uniform(3, 15, dsm.shape)
    dsm[building_mask] += building_heights[building_mask]
    
    # Small vegetation in grass areas (0.5-2m high)
    grass_heights = np.random.uniform(0.5, 2, dsm.shape)
    dsm[grass_mask] += grass_heights[grass_mask]
    
    # Create DEM (ground elevation without objects)
    dem = base_elevation.copy()
    
    # Add band dimensions
    dsm = dsm[np.newaxis, :, :]
    dem = dem[np.newaxis, :, :]
    
    # Create dummy transform
    transform = rasterio.transform.from_bounds(0, 0, 10, 10, width, height)
    
    print(f"   ✅ Created synthetic data:")
    print(f"      RGBI shape: {rgbi.shape}")
    print(f"      DSM shape: {dsm.shape}")
    print(f"      DEM shape: {dem.shape}")
    print(f"      Forest area: {forest_mask.sum():,} pixels")
    print(f"      Grass area: {grass_mask.sum():,} pixels")
    print(f"      Urban area: {urban_mask.sum():,} pixels")
    
    return rgbi, dsm, dem, transform

# Create the sample data
rgbi, dsm, dem, transform = create_sample_data()

## 8. Complete Processing Example

In [ ]:
# Initialize the processor
processor = NDVIProcessor()
processor.set_debug_mode(True)
processor.set_height_usage(True)

print("\n🚀 Starting complete NDVI processing workflow...")

# Step 1: Calculate NDVI
ndvi = processor.calculate_ndvi(rgbi)
print(f"\n📊 NDVI calculation complete!")

# Step 2: Calculate nDSM (height above ground)
ndsm = processor.calculate_ndsm(dsm, dem)

# Step 3: Calculate slope
slope = processor.calculate_slope(dem, transform)
slope_classification = processor.classify_slope(slope, threshold=5.0)

# Step 4: Classify vegetation
vegetation_classification = processor.classify_vegetation(
    ndvi, ndsm, 
    ndvi_threshold=0.3, 
    height_threshold=3.5,
    max_height=50.0
)

# Step 5: Apply morphological cleaning
cleaned_classification = processor.apply_morphological_cleaning(
    vegetation_classification, iterations=2
)

print("\n✅ Processing workflow complete!")

## 9. Visualize Results

In [ ]:
# Create comprehensive visualizations
print("📊 Creating visualization plots...")

# Display results
visualize_results(rgbi, ndvi, cleaned_classification, ndsm, slope)

# Print detailed statistics
create_statistics_summary(ndvi, cleaned_classification, ndsm)

print("\n🎨 Visualization complete!")

## 10. Save Results Function

In [ ]:
def save_results(output_dir: str, ndvi: np.ndarray, classification: np.ndarray, 
                ndsm: np.ndarray, slope: np.ndarray, transform: rasterio.Affine):
    """Save processing results to GeoTIFF files."""
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    print(f"💾 Saving results to: {output_dir}")
    
    # Common raster properties
    height, width = ndvi.shape
    crs = 'EPSG:2193'  # NZGD2000 / New Zealand Transverse Mercator 2000
    
    # Save NDVI
    ndvi_path = os.path.join(output_dir, 'ndvi.tif')
    with rasterio.open(
        ndvi_path, 'w',
        driver='GTiff',
        height=height, width=width, count=1,
        dtype=ndvi.dtype, crs=crs, transform=transform,
        compress='lzw'
    ) as dst:
        dst.write(ndvi, 1)
    print(f"   ✅ NDVI saved: {ndvi_path}")
    
    # Save classification
    class_path = os.path.join(output_dir, 'vegetation_classification.tif')
    with rasterio.open(
        class_path, 'w',
        driver='GTiff',
        height=height, width=width, count=1,
        dtype=classification.dtype, crs=crs, transform=transform,
        compress='lzw'
    ) as dst:
        dst.write(classification, 1)
    print(f"   ✅ Classification saved: {class_path}")
    
    # Save nDSM
    ndsm_path = os.path.join(output_dir, 'ndsm.tif')
    with rasterio.open(
        ndsm_path, 'w',
        driver='GTiff',
        height=height, width=width, count=1,
        dtype=ndsm.dtype, crs=crs, transform=transform,
        compress='lzw'
    ) as dst:
        dst.write(ndsm, 1)
    print(f"   ✅ nDSM saved: {ndsm_path}")
    
    # Save slope
    slope_path = os.path.join(output_dir, 'slope.tif')
    with rasterio.open(
        slope_path, 'w',
        driver='GTiff',
        height=height, width=width, count=1,
        dtype=slope.dtype, crs=crs, transform=transform,
        compress='lzw'
    ) as dst:
        dst.write(slope, 1)
    print(f"   ✅ Slope saved: {slope_path}")
    
    print(f"\n🎉 All results saved successfully!")

# Save the processing results
save_results('ndvi_processing_results', ndvi, cleaned_classification, ndsm, slope, transform)

## Summary

This notebook demonstrated comprehensive vegetation analysis using NDVI and multi-sensor data:

### ✅ **Key Accomplishments:**

🌿 **NDVI calculation** from RGBI imagery with proper band handling  
🏔️ **Height analysis** using DSM and DEM for object height extraction  
⛰️ **Slope analysis** using Horn's method for terrain characterization  
🎯 **Multi-criteria classification** combining spectral and height information  
🧹 **Morphological cleaning** to improve classification results  
📊 **Comprehensive visualization** of all processing steps  
💾 **Results export** to standard GeoTIFF format  

### 🔬 **Scientific Methods Used:**

- **NDVI**: `(NIR - Red) / (NIR + Red)` for vegetation mapping
- **nDSM**: `DSM - DEM` for object height above ground
- **Horn's slope algorithm**: For terrain slope calculation
- **Multi-threshold classification**: Combining vegetation index and height
- **Morphological operations**: For spatial filtering and noise reduction

### 🎯 **Applications:**

- **Forestry management**: Tree height and canopy analysis
- **Urban planning**: Green space mapping and analysis
- **Agriculture**: Crop health monitoring and yield estimation
- **Environmental monitoring**: Habitat mapping and change detection
- **Carbon storage**: Biomass estimation using height and vegetation data

### 🚀 **Next Steps:**

- Apply to real LINZ imagery data downloaded from previous notebooks
- Experiment with different NDVI thresholds for your study area
- Integrate temporal analysis for change detection
- Combine with vector data for detailed landscape analysis
- Develop automated processing pipelines for large datasets